In [1]:
import requests
import pandas as pd
import os
from pathlib import Path

# Paths based on structure
PROJECT_DIR = Path("..")
REF_DIR = PROJECT_DIR / "data" / "reference"
YEARLY_DIR = PROJECT_DIR / "outputs" / "yearly"

REF_DIR.mkdir(parents=True, exist_ok=True)

print("Reference directory:", REF_DIR.resolve())
print("Yearly parquet directory:", YEARLY_DIR.resolve())

Reference directory: /Users/francobastida/Desktop/ecobici/data/reference
Yearly parquet directory: /Users/francobastida/Desktop/ecobici/outputs/yearly


In [2]:
# GBFS API pull, since there is no historical data
GBFS_ROOT_URL = "https://gbfs.mex.lyftbikes.com/gbfs/gbfs.json"

response = requests.get(GBFS_ROOT_URL, timeout=30)
response.raise_for_status()

gbfs_root = response.json()

In [3]:
gbfs_root

{'last_updated': 1774139317,
 'ttl': 10,
 'data': {'en': {'feeds': [{'name': 'station_status',
     'url': 'https://gbfs.mex.lyftbikes.com/gbfs/en/station_status.json'},
    {'name': 'system_information',
     'url': 'https://gbfs.mex.lyftbikes.com/gbfs/en/system_information.json'},
    {'name': 'station_information',
     'url': 'https://gbfs.mex.lyftbikes.com/gbfs/en/station_information.json'},
    {'name': 'system_alerts',
     'url': 'https://gbfs.mex.lyftbikes.com/gbfs/en/system_alerts.json'},
    {'name': 'free_bike_status',
     'url': 'https://gbfs.mex.lyftbikes.com/gbfs/en/free_bike_status.json'}]},
  'es': {'feeds': [{'name': 'station_status',
     'url': 'https://gbfs.mex.lyftbikes.com/gbfs/es/station_status.json'},
    {'name': 'system_information',
     'url': 'https://gbfs.mex.lyftbikes.com/gbfs/es/system_information.json'},
    {'name': 'station_information',
     'url': 'https://gbfs.mex.lyftbikes.com/gbfs/es/station_information.json'},
    {'name': 'system_alerts',
   

In [4]:
# Extract station_information URL
feeds = gbfs_root["data"]["en"]["feeds"]
feeds_df = pd.DataFrame(feeds)

station_info_url = feeds_df.loc[
    feeds_df["name"] == "station_information", "url"
].iloc[0]

# Download station info
station_response = requests.get(station_info_url, timeout=30)
station_response.raise_for_status()

station_json = station_response.json()

In [5]:
station_json

{'last_updated': 1774139317,
 'ttl': 10,
 'data': {'stations': [{'station_id': '1',
    'external_id': 'e961269c-34c4-4b70-8e30-a51aa95a8429',
    'name': 'CE-710 Molino del Rey - Glorieta de la Lealtad',
    'short_name': '710',
    'lat': 19.416795,
    'lon': -99.192508,
    'rental_methods': ['KEY', 'CREDITCARD'],
    'capacity': 39,
    'electric_bike_surcharge_waiver': False,
    'is_charging': False,
    'eightd_has_key_dispenser': False,
    'has_kiosk': True},
   {'station_id': '5',
    'external_id': '3ea89109-d2f3-46eb-9c41-c43742050340',
    'name': 'CE-407  Prolongación Xochicalco-General Emiliano Zapata',
    'short_name': '407',
    'lat': 19.36726591257277,
    'lon': -99.15865559132271,
    'rental_methods': ['KEY', 'CREDITCARD'],
    'capacity': 19,
    'electric_bike_surcharge_waiver': False,
    'is_charging': False,
    'eightd_has_key_dispenser': False,
    'has_kiosk': True},
   {'station_id': '6',
    'external_id': 'ba78b703-4e5a-44bd-ab2c-1eedc71e11c3',
    'n

In [6]:
# Data frame and inspect
stations_raw = pd.DataFrame(station_json["data"]["stations"])

print("Shape:", stations_raw.shape)
stations_raw.head()
print(stations_raw.columns.tolist())

Shape: (677, 12)
['station_id', 'external_id', 'name', 'short_name', 'lat', 'lon', 'rental_methods', 'capacity', 'electric_bike_surcharge_waiver', 'is_charging', 'eightd_has_key_dispenser', 'has_kiosk']


In [7]:
stations_raw

,station_id,external_id,name,short_name,lat,lon,rental_methods,capacity,electric_bike_surcharge_waiver,is_charging,eightd_has_key_dispenser,has_kiosk
0,1,e961269c-34c4-4b70-8e30-a51aa95a8429,CE-710 Molino del Rey - Glorieta de la Lealtad,710,19.416795,-99.192508,"[KEY, CREDITCARD]",39,False,False,False,True
1,5,3ea89109-d2f3-46eb-9c41-c43742050340,CE-407 Prolongación Xochicalco-General Emilia...,407,19.367266,-99.158656,"[KEY, CREDITCARD]",19,False,False,False,True
2,6,ba78b703-4e5a-44bd-ab2c-1eedc71e11c3,CE-428 Prolongación Uxmal-Av. Popocatépetl (E...,428,19.363404,-99.160395,"[KEY, CREDITCARD]",27,False,False,False,True
3,7,6563d263-2342-46e3-8983-461e68d2d615,CE-427 Avenida México-Coyoacán-Av. Popocatépet...,427,19.364906,-99.162987,"[KEY, CREDITCARD]",19,False,False,False,True
4,8,ec55e597-c8fc-4e86-bcfe-b0e81a494790,CE-443 Bruno Traven-Golondrinas,443,19.359583,-99.162085,"[KEY, CREDITCARD]",31,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
672,697,ea8b036f-bfde-4aa4-8bc0-57405c27fd8a,CE-511 Palestina - Egipto,511,19.463941,-99.182280,"[KEY, CREDITCARD]",27,False,False,False,True
673,698,1f652ea2-4be5-4d30-b73c-2cb7d1266a70,CE-682 Manuel M. Ponce - Fernando M. Villalpando,682,19.353534,-99.189548,"[KEY, CREDITCARD]",23,False,False,False,True
674,699,a1e9daf9-b993-4abb-8e39-66515ffcfa56,CE-681 Guty Cárdenas - Ricardo Castro,681,19.355844,-99.189141,"[KEY, CREDITCARD]",19,False,False,False,True
675,700,09fac974-dc65-4213-b3e6-c5b7440f37df,CE-711 Molino del Rey - Av. Constituyentes,711,19.414108,-99.191983,"[KEY, CREDITCARD]",39,False,False,False,True


- We have 677 active stations as of March 19, 2025
- Short name is the identifier that could be merged

In [8]:
# Define station characteristics 
station_characteristics = [
    c for c in [
        "station_id",
        "name",
        "short_name",
        "lat",
        "lon",
        "capacity",
        "region_id",
        "address"
    ]
    if c in stations_raw.columns
]

stations_ref = stations_raw[station_characteristics].copy()

# Standardize station_id for potential merges
stations_ref["station_id"] = stations_ref["station_id"].astype(str).str.strip()

# Remove duplicate station_ids if any
stations_ref = stations_ref.drop_duplicates(subset=["station_id"])

print("Clean station reference shape:", stations_ref.shape)
stations_ref.head()

Clean station reference shape: (677, 6)


,station_id,name,short_name,lat,lon,capacity
0,1,CE-710 Molino del Rey - Glorieta de la Lealtad,710,19.416795,-99.192508,39
1,5,CE-407 Prolongación Xochicalco-General Emilia...,407,19.367266,-99.158656,19
2,6,CE-428 Prolongación Uxmal-Av. Popocatépetl (E...,428,19.363404,-99.160395,27
3,7,CE-427 Avenida México-Coyoacán-Av. Popocatépet...,427,19.364906,-99.162987,19
4,8,CE-443 Bruno Traven-Golondrinas,443,19.359583,-99.162085,31


In [9]:
# Quality check
print("Missing lat:", stations_ref["lat"].isna().sum())
print("Missing lon:", stations_ref["lon"].isna().sum())

# Save
csv_path = REF_DIR / "stations_gbfs_current.csv"
parquet_path = REF_DIR / "stations_gbfs_current.parquet"

stations_ref.to_csv(csv_path, index=False)
stations_ref.to_parquet(parquet_path, index=False)

print("Saved:")
print(csv_path.resolve())
print(parquet_path.resolve())

Missing lat: 0
Missing lon: 0
Saved:
/Users/francobastida/Desktop/ecobici/data/reference/stations_gbfs_current.csv
/Users/francobastida/Desktop/ecobici/data/reference/stations_gbfs_current.parquet


In [10]:
#Check parquet
stations_ref = pd.read_parquet(REF_DIR / "stations_gbfs_current.parquet")
stations_ref

,station_id,name,short_name,lat,lon,capacity
0,1,CE-710 Molino del Rey - Glorieta de la Lealtad,710,19.416795,-99.192508,39
1,5,CE-407 Prolongación Xochicalco-General Emilia...,407,19.367266,-99.158656,19
2,6,CE-428 Prolongación Uxmal-Av. Popocatépetl (E...,428,19.363404,-99.160395,27
3,7,CE-427 Avenida México-Coyoacán-Av. Popocatépet...,427,19.364906,-99.162987,19
4,8,CE-443 Bruno Traven-Golondrinas,443,19.359583,-99.162085,31
...,...,...,...,...,...,...
672,697,CE-511 Palestina - Egipto,511,19.463941,-99.182280,27
673,698,CE-682 Manuel M. Ponce - Fernando M. Villalpando,682,19.353534,-99.189548,23
674,699,CE-681 Guty Cárdenas - Ricardo Castro,681,19.355844,-99.189141,19
675,700,CE-711 Molino del Rey - Av. Constituyentes,711,19.414108,-99.191983,39


In [11]:
def get_active_stations(year: int) -> pd.DataFrame:
    df = pd.read_parquet(YEARLY_DIR / f"ecobici_trips_{year}.parquet")

    start_ids = df["start_station_id"].dropna().astype("Int64").astype(str)
    end_ids = df["end_station_id"].dropna().astype("Int64").astype(str)

    active_ids = pd.concat([start_ids, end_ids], ignore_index=True).drop_duplicates()

    return pd.DataFrame({"station_id": active_ids})

In [12]:
def get_station_map_df(year: int) -> pd.DataFrame:
    stations_year = get_active_stations(year)
    stations_year["station_id"] = stations_year["station_id"].astype(str).str.strip()

    merged = stations_year.merge(
        stations_ref,
        on="station_id",
        how="left"
    )

    print(f"\nYear {year}")
    print("Active stations:", len(merged))
    print("Total stations:", len(merged))
    print("Missing lat:", merged["lat"].isna().sum())
    print("Missing lng:", merged["lon"].isna().sum())

    return merged

In [18]:
import folium

def plot_stations_map(df: pd.DataFrame, title: str):
    # center on Mexico City
    m = folium.Map(
        location=[19.43, -99.13],
        zoom_start=12,
        tiles="CartoDB positron"
    )
    for _, row in df.dropna(subset=["lat", "lon"]).iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=2,
            color="blue",
            fill=True,
            fill_opacity=0.6,
            popup=str(row.get("station_id", ""))
        ).add_to(m)

    return m

In [14]:
df_2018 = get_station_map_df(2018)
df_2025 = get_station_map_df(2025)


Year 2018
Active stations: 482
Total stations: 482
Missing lat: 13
Missing lng: 13

Year 2025
Active stations: 666
Total stations: 666
Missing lat: 34
Missing lng: 34


In [20]:
map_2018 = plot_stations_map(df_2018, "Stations active in 2018")
map_2018
map_2018.save("../outputs/map_2018.html")

In [22]:
map_2025 = plot_stations_map(df_2025, "Stations active in 2025")
map_2025
map_2025.save("../outputs/map_2025.html")